In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import from_json, col, explode,to_date,regexp_replace,trim,length,desc,current_timestamp,max as spark_max
from pyspark.sql.types import StructType, StructField, StringType, ArrayType,FloatType
from delta import configure_spark_with_delta_pip,DeltaTable

In [2]:
spark_builder = (
    SparkSession.builder
    .appName("BronzetoSilver")
    .master("local[*]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)
spark = configure_spark_with_delta_pip(spark_builder).getOrCreate()

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-e2d97938-0368-478b-b4c3-9475c1da4e58;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.1.0 in central
	found io.delta#delta-storage;3.1.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 218ms :: artifacts dl 13ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.1.0 from central in [default]
	io.delta#delta-storage;3.1.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0   |   0   |  

In [4]:
# Check if silver table exists
silver_table_path = "/opt/spark/data/silver/invoice_ocr/"
if DeltaTable.isDeltaTable(spark, silver_table_path):
    last_ingestion_ts = (
        spark.read.format("delta")
        .load(silver_path)
        .select(spark_max("ingestion_ts").alias("max_ts"))
        .collect()[0]["max_ts"]
    )
else:
    last_ingestion_ts = None
print(last_ingestion_ts)

26/02/09 09:16:20 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
[Stage 9:>                                                          (0 + 2) / 2]

2026-02-09 08:57:46.127460


In [5]:
bronze_df = spark.read.format("delta").load("/opt/spark/data/bronze/invoices_raw")

spark.conf.set("spark.sql.debug.maxToStringFields", 1000)  # default is 25


In [6]:
if last_ingestion_ts:
    bronze_incremental_df = bronze_df.filter(
        col("ingestion_ts") > last_ingestion_ts
    )
else:
    bronze_incremental_df = bronze_df


In [7]:
bronze_incremental_df.show(5)

+---------------+--------------------+--------------------+------------+--------------------+
|      file_name|           json_data|          ocred_text| source_file|        ingestion_ts|
+---------------+--------------------+--------------------+------------+--------------------+
|batch1-1142.jpg|\n{\n  "invoice":...|Invoice no: 96556...|batch1_3.csv|2026-02-09 09:15:...|
|batch1-1147.jpg|\n{\n  "invoice":...|Invoice no: 95894...|batch1_3.csv|2026-02-09 09:15:...|
|batch1-1080.jpg|\n{\n  "invoice":...|Invoice no: 17585...|batch1_3.csv|2026-02-09 09:15:...|
|batch1-1146.jpg|\n{\n  "invoice":...|Invoice no: 27702...|batch1_3.csv|2026-02-09 09:15:...|
|batch1-1090.jpg|\n{\n  "invoice":...|Invoice no: 34310...|batch1_3.csv|2026-02-09 09:15:...|
+---------------+--------------------+--------------------+------------+--------------------+
only showing top 5 rows



In [8]:
json_schema = StructType([
    StructField("invoice", StructType([
        StructField("client_name", StringType(), True),
        StructField("client_address", StringType(), True),
        StructField("seller_name", StringType(), True),
        StructField("seller_address", StringType(), True),
        StructField("invoice_number", StringType(), True),
        StructField("invoice_date", StringType(), True),
        StructField("due_date", StringType(), True)
    ]), True),
    StructField("items", ArrayType(
        StructType([
            StructField("description", StringType(), True),
            StructField("quantity", StringType(), True),
            StructField("total_price", StringType(), True)
        ])
    ), True),
    StructField("subtotal", StructType([
        StructField("tax", StringType(), True),
        StructField("discount", StringType(), True),
        StructField("total", StringType(), True)
    ]), True),
    StructField("payment_instructions", StructType([
        StructField("due_date", StringType(), True),
        StructField("bank_name", StringType(), True),
        StructField("account_number", StringType(), True),
        StructField("payment_method", StringType(), True)
    ]), True)
])

In [9]:
# Parse JSON column (assuming the column is 'json_col')
parsed_df = bronze_incremental_df.withColumn("json_data", from_json(col("json_data"), json_schema))

In [10]:
parsed_df.show(5)

+---------------+--------------------+--------------------+------------+--------------------+
|      file_name|           json_data|          ocred_text| source_file|        ingestion_ts|
+---------------+--------------------+--------------------+------------+--------------------+
|batch1-1142.jpg|{{Miller PLC, 401...|Invoice no: 96556...|batch1_3.csv|2026-02-09 09:15:...|
|batch1-1147.jpg|{{Cruz, Miller an...|Invoice no: 95894...|batch1_3.csv|2026-02-09 09:15:...|
|batch1-1080.jpg|{{Carey-Montgomer...|Invoice no: 17585...|batch1_3.csv|2026-02-09 09:15:...|
|batch1-1146.jpg|{{Moore PLC, 928 ...|Invoice no: 27702...|batch1_3.csv|2026-02-09 09:15:...|
|batch1-1090.jpg|{{Thomas-Lee, 851...|Invoice no: 34310...|batch1_3.csv|2026-02-09 09:15:...|
+---------------+--------------------+--------------------+------------+--------------------+
only showing top 5 rows



In [11]:
exploded_df = parsed_df.withColumn("item", explode(col("json_data.items")))

In [12]:
from pyspark.sql.functions import col, udf
from pyspark.sql.types import FloatType
import re

# Function to normalize numeric strings
def parse_numeric(x):
    if x is None:
        return None
    x = str(x).strip()
    x = x.replace(" ", "")

    # Remove thousand separator (commas before digits and dot)
    # European style: 57,16 -> 57.16
    # US style: 1,234.56 -> 1234.56
    # Detect if last comma is decimal separator
    if ',' in x:
        if re.match(r'^\d{1,3}(,\d{3})*(\.\d+)?$', x):  # US style with thousand separator
            x = x.replace(',', '')
        else:  # European style, comma is decimal
            x = x.replace(',', '.')
    try:
        return float(x)
    except:
        return None

# Register UDF
parse_numeric_udf = udf(parse_numeric, FloatType())


In [13]:
silver_df = exploded_df.select(
    col("json_data.invoice.invoice_number").alias("invoice_number"),
    to_date(col("json_data.invoice.invoice_date"), "MM/dd/yyyy").alias("invoice_date"),
    col("json_data.invoice.client_name").alias("client_name"),
    col("json_data.invoice.client_address").alias("client_address"),
    col("json_data.invoice.seller_name").alias("seller_name"),
    col("json_data.invoice.seller_address").alias("seller_address"),
    col("item.description").alias("item_description"),
    parse_numeric_udf(col("item.quantity")).cast(FloatType()).alias("item_quantity"),
    parse_numeric_udf(col("item.total_price")).cast(FloatType()).alias("item_total_price"),
    parse_numeric_udf(col("json_data.subtotal.tax")).cast(FloatType()).alias("tax"),
    parse_numeric_udf(col("json_data.subtotal.discount")).cast(FloatType()).alias("discount"),
    parse_numeric_udf(col("json_data.subtotal.total")).cast(FloatType()).alias("total"),
    col("file_name").alias("invoice_image_name")
                     )

In [14]:
silver_df.show(5,truncate=False)

[Stage 22:>                                                         (0 + 1) / 1]

+--------------+------------+-----------+-----------------------------------------------------+-------------------------+--------------------------------+-------------------------------------------------------------------------+-------------+----------------+-----+--------+-------+------------------+
|invoice_number|invoice_date|client_name|client_address                                       |seller_name              |seller_address                  |item_description                                                         |item_quantity|item_total_price|tax  |discount|total  |invoice_image_name|
+--------------+------------+-----------+-----------------------------------------------------+-------------------------+--------------------------------+-------------------------------------------------------------------------+-------------+----------------+-----+--------+-------+------------------+
|96556696      |2016-02-25  |Miller PLC |401 Leblanc Isle Suite 451\nPort Kellimouth, NJ 99074

In [24]:
silver_df.count()

1724

In [16]:
## Removing trialing spaces and new line characters for string types

# List of string columns
string_cols = [f.name for f in silver_df.schema.fields if isinstance(f.dataType, StringType)]

# Start with original DataFrame
df_silver_cleaned = silver_df

# Apply cleaning to all string columns cumulatively
for c in string_cols:
    
    df_silver_cleaned = df_silver_cleaned.withColumn(c, trim(regexp_replace(regexp_replace(col(c), r"[\n\r]+", " "),r"[^a-zA-Z0-9\s\.\-]", "")))

# Display nicely
df_silver_cleaned = df_silver_cleaned.withColumn("ingestion_ts", current_timestamp())
df_silver_cleaned.show(20, truncate=False)

+--------------+------------+---------------------+---------------------------------------------------+--------------------------+------------------------------------------+--------------------------------------------------------------------------------+-------------+----------------+------+--------+--------+------------------+--------------------------+
|invoice_number|invoice_date|client_name          |client_address                                     |seller_name               |seller_address                            |item_description                                                                |item_quantity|item_total_price|tax   |discount|total   |invoice_image_name|ingestion_ts              |
+--------------+------------+---------------------+---------------------------------------------------+--------------------------+------------------------------------------+--------------------------------------------------------------------------------+-------------+----------------+-

In [17]:
df_silver_cleaned.count()

1904

In [18]:
spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")

In [19]:
# Check if table exists
if DeltaTable.isDeltaTable(spark, silver_table_path):
    silver_delta = DeltaTable.forPath(spark, silver_table_path)
    
    # Merge into existing Silver table with schema evolution
    silver_delta.alias("tgt").merge(
        df_silver_cleaned.alias("src"),
        "tgt.invoice_number = src.invoice_number AND tgt.item_description = src.item_description"
    ).whenMatchedUpdate(set={
        "invoice_date": "src.invoice_date",
        "client_name": "src.client_name",
        "client_address": "src.client_address",
        "seller_name": "src.seller_name",
        "seller_address": "src.seller_address",
        "item_quantity": "src.item_quantity",
        "item_total_price": "src.item_total_price",
        "tax": "src.tax",
        "discount": "src.discount",
        "total": "src.total",
        "invoice_image_name": "src.invoice_image_name",
        "ingestion_ts": "src.ingestion_ts"
    }).whenNotMatchedInsertAll().execute()

else:
    # If table doesn't exist, create it with mergeSchema enabled
    df_silver_cleaned.write.format("delta") \
        .option("mergeSchema", "true") \
        .mode("overwrite") \
        .save(silver_table_path)

In [20]:
df_new_silver = spark.read.option("format","delta").load("/opt/spark/data/silver/invoice_ocr/")

In [21]:
df_new_silver.printSchema()

root
 |-- invoice_number: string (nullable = true)
 |-- invoice_date: date (nullable = true)
 |-- client_name: string (nullable = true)
 |-- client_address: string (nullable = true)
 |-- seller_name: string (nullable = true)
 |-- seller_address: string (nullable = true)
 |-- item_description: string (nullable = true)
 |-- item_quantity: float (nullable = true)
 |-- item_total_price: float (nullable = true)
 |-- tax: float (nullable = true)
 |-- discount: float (nullable = true)
 |-- total: float (nullable = true)
 |-- invoice_image_name: string (nullable = true)
 |-- ingestion_ts: timestamp (nullable = true)



In [22]:
df_new_silver.createOrReplaceTempView("invoices_silver")

In [23]:
## Data quality check 
spark.sql(""" Select count(*) from invoices_silver where total is not NULL""").show()

+--------+
|count(1)|
+--------+
|    5602|
+--------+



In [24]:
spark.stop()